# Week 3: Named Entity Recognition (NER)

606363 Computing Systems for Data Science and AI — every code example from the lesson page, in order.

Lesson page: https://zmomani.github.io/ComputingSystemsDS-AI/lessons/week03-ner.html

**How to use:** open this notebook in the course codespace (folder `notebooks/`) and run the cells from the top (Shift+Enter, or Run All). Later examples often reuse variables from earlier ones. Examples that call OpenAI or other APIs need your key as a Codespaces secret or in the `.env` file. For Ollama examples first run `bash scripts/start-ollama.sh` in the terminal. Model-training examples are much faster on a GPU: open this notebook in Google Colab and choose Runtime → Change runtime type → T4 GPU (in Colab add API keys with the 🔑 Secrets panel).

## Run in your browser

**From BIO tags to entity spans, and entity-level F1**

In [ ]:
tokens = ["Dr.", "Albanna", "teaches", "at", "the", "University", "of", "Petra", "in", "Amman"]
gold   = ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "I-ORG", "O", "B-LOC"]
pred   = ["B-PER", "I-PER", "O", "O", "O", "B-ORG", "I-ORG", "B-LOC", "O", "B-LOC"]

def spans(tags):
    out, cur = [], None
    for i, t in enumerate(tags):
        if t.startswith("B-"):
            if cur: out.append(cur)
            cur = [t[2:], i, i]
        elif t.startswith("I-") and cur and cur[0] == t[2:]:
            cur[2] = i
        else:
            if cur: out.append(cur)
            cur = None
    if cur: out.append(cur)
    return {(typ, s, e) for typ, s, e in out}

G, P = spans(gold), spans(pred)
for typ, s, e in sorted(G | P, key=lambda x: x[1]):
    mark = "correct" if (typ, s, e) in G and (typ, s, e) in P else ("missed" if (typ, s, e) in G else "spurious")
    print(f"{typ:4} {' '.join(tokens[s:e+1]):28} {mark}")

tp = len(G & P)
precision, recall = tp / len(P), tp / len(G)
f1 = 2 * precision * recall / (precision + recall)
token_acc = sum(g == p for g, p in zip(gold, pred)) / len(gold)
print(f"\ntoken accuracy = {token_acc:.2f}, entity P={precision:.2f} R={recall:.2f} F1={f1:.2f}")